In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
data_path = Path("..") / "data" / "credit_risk_dataset.csv"
if not data_path.exists():
    data_path = Path("data") / "credit_risk_dataset.csv"
df = pd.read_csv(data_path)
# df.head()
# df.info()
# df.describe()
# df.isnull().sum()

missing = df.isnull().sum()

missing_summary = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": (missing / len(df)) * 100
})

missing_summary[missing_summary["Missing_Count"] > 0]

# df[["person_emp_length", "loan_int_rate"]].median()
# df.describe().T

# print("Applicants aged above 100:")
# display(df[df["person_age"] > 100][["person_age", "person_income", "loan_status"]])

# print("Employment length above 50 years:")
# display(df[df["person_emp_length"] > 50][["person_age", "person_emp_length"]])

# print("Income above 1 million:")
# display(df[df["person_income"] > 1_000_000][["person_age", "person_income"]])

suspicious_rows = df[
    (df["person_age"] > 100) |
    (df["person_emp_length"] > 50)
]

# display(suspicious_rows)

# Replace unrealistic ages with NaN
df.loc[df["person_age"] > 100, "person_age"] = float("nan")

# Replace unrealistic employment lengths with NaN
df.loc[df["person_emp_length"] > 50, "person_emp_length"] = float("nan")

# print("Missing values after cleaning:")
# print(df[["person_age", "person_emp_length"]].isnull().sum())

df[[
    "person_age",
    "person_income",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_cred_hist_length"
]].skew()

# plt.figure(figsize=(14, 5))

# plt.subplot(1, 2, 1)
# sns.histplot(df["person_income"], bins=50, kde=True)
# plt.title("Income Distribution - Original")
# plt.xlabel("Annual Income")

# plt.subplot(1, 2, 2)
# sns.histplot(df["person_income"], bins=50, kde=True)
# plt.xscale("log")
# plt.title("Income Distribution - Log X-axis")
# plt.xlabel("Annual Income (Log Scale)")

# plt.tight_layout()
# plt.show()

percentiles = [25, 50, 75, 90, 95, 99, 100]

df["person_income"].quantile(
[p / 100 for p in percentiles]
)

Q1 = df["person_income"].quantile(0.25)
Q3 = df["person_income"].quantile(0.75)

IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

# print("IQR:", IQR)
# print("Upper Bound:", upper_bound)

# print(
#     "Potential high outliers:",
#     (df["person_income"] > upper_bound).sum()
# )

high_income = df[df["person_income"] > upper_bound]

high_income["person_income"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

df.nlargest(
    10, "person_income"
)[[
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_status",
    "loan_percent_income"
]]

df["calculated_loan_percent"] = (
    df["loan_amnt"] / df["person_income"]
)

df[[
    "loan_amnt",
    "person_income",
    "loan_percent_income",
    "calculated_loan_percent"
]].head(10)

df["calculated_loan_percent"] = (
    df["loan_amnt"] / df["person_income"]
)

df["loan_percent_difference"] = (
    abs(
        df["loan_percent_income"]
        - df["calculated_loan_percent"]
    )
)

mismatches = df[
    df["loan_percent_difference"] > 0.015
]

# print("Total mismatches:", len(mismatches))

mismatches[[
    "person_income",
    "loan_amnt",
    "loan_percent_income",
    "calculated_loan_percent",
    "loan_percent_difference"
]].head(10)

# print("Total mismatches:", len(mismatches))

# print(
#     "Percentage of dataset:",
#     round(len(mismatches) / len(df) * 100, 2),
#     "%"
# )

# print(
#     mismatches["loan_percent_difference"].describe()
# )

mismatches["loan_amnt"].value_counts().head(10)
mismatches["loan_status"].value_counts()

df.groupby("loan_status")["loan_percent_income"].agg(
    ["mean", "median", "min", "max"]
)

df["income_group"] = pd.cut(
    df["loan_percent_income"],
    bins=[0, 0.10, 0.20, 0.30, 1],
    labels=["0-10%", "10-20%", "20-30%", "Above 30%"],
    include_lowest=True
)

df["income_group"].value_counts().sort_index()

df.groupby("income_group")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df["interest_group"] = pd.cut(
    df["loan_int_rate"],
    bins=[0, 8, 12, 16, float("inf")],
    labels=["Below 8%", "8-12%", "12-16%", "Above 16%"]
)

df["interest_group"].value_counts().sort_index()

df.groupby("interest_group")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df["employment_group"] = pd.cut(
    df["person_emp_length"],
    bins=[-1, 2, 5, 10, float("inf")],
    labels=["0-2 years", "2-5 years", "5-10 years", "Above 10 years"]
)

df["employment_group"].value_counts().sort_index()

df.groupby("employment_group")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df["credit_history_group"] = pd.cut(
    df["cb_person_cred_hist_length"],
    bins=[0, 3, 5, 10, float("inf")],
    labels=["0-3 years", "3-5 years", "5-10 years", "Above 10 years"],
    include_lowest=True
)

df["credit_history_group"].value_counts().sort_index()

df.groupby("credit_history_group")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df.groupby("loan_grade")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df.groupby("person_home_ownership")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df.groupby("loan_intent")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df.groupby("cb_person_default_on_file")["loan_status"].agg(
    ["count", "sum", "mean"]
)

df.groupby("loan_grade")["loan_int_rate"].agg(["count", "mean", "median"])

df["loan_int_rate"].isna().sum()

df.groupby("loan_grade")["loan_int_rate"].apply(
    lambda s: s.isna().sum()
)

df.groupby("loan_grade")["loan_int_rate"].apply(
    lambda s: s.isna().mean() * 100
).round(2)

df.groupby(df["loan_int_rate"].isna())["loan_status"].agg(
    ["count", "sum", "mean"]
)

df.duplicated().sum()

duplicates = df[df.duplicated(keep=False)]
duplicates["loan_status"].value_counts()

df.loc[df.duplicated(), "loan_status"].value_counts()
df["loan_status"].value_counts(normalize=True).mul(100).round(2)
df.columns.tolist()

original_cols = [
    "person_age", "person_income", "person_home_ownership",
    "person_emp_length", "loan_intent", "loan_grade",
    "loan_amnt", "loan_int_rate", "loan_status",
    "loan_percent_income", "cb_person_default_on_file",
    "cb_person_cred_hist_length"
]

df.duplicated(subset=original_cols).sum()
df["person_age"].describe()

(df["person_age"] > 80).sum()

df.loc[df["person_age"] > 80, original_cols]

df["person_income"].describe()
high_income = df[df["person_income"] > 500_000]

print("Rows over $500,000:", len(high_income))
high_income[["person_age", "person_income", "loan_amnt", "loan_status"]]
high_income.nlargest(10, "person_income")[
    ["person_age", "person_income", "loan_amnt", "loan_status"]
]

high_income.nlargest(10, "person_income")[
    ["person_age", "person_income", "loan_amnt",
     "loan_percent_income", "loan_status"]
]

df["person_emp_length"].describe()
df.loc[
    df["person_emp_length"] > 30,
    ["person_age", "person_emp_length", "person_income", "loan_status"]
]

df["loan_amnt"].describe()

df.isna().sum().sort_values(ascending=False).head(10)

from sklearn.model_selection import train_test_split

features = [
    "person_age",
    "person_income",
    "person_home_ownership",
    "person_emp_length",
    "loan_intent",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_default_on_file",
    "cb_person_cred_hist_length",
]

X = df[features]
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_cred_hist_length",
]

categorical_features = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test, y_pred, target_names=["No default", "Default"]
))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_probability))

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

model.fit(X_fit, y_fit)
val_probabilities = model.predict_proba(X_val)[:, 1]

for threshold in [0.5, 0.4, 0.3, 0.2]:
    val_predictions = (val_probabilities >= threshold).astype(int)
    print(
        f"Threshold {threshold}: "
        f"precision={precision_score(y_val, val_predictions):.2f}, "
        f"recall={recall_score(y_val, val_predictions):.2f}"
    )

model.fit(X_train, y_train)

test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.3).astype(int)

print(classification_report(
    y_test, test_predictions, target_names=["No default", "Default"]
))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

import pandas as pd
def score_applicant(applicant_data):
    applicant_df = pd.DataFrame([applicant_data], columns=features)
    probability = model.predict_proba(applicant_df)[0, 1]
    prediction = "Default" if probability >= 0.3 else "No default"

    return {
        "default_probability": round(float(probability), 3),
        "prediction": prediction,
    }

score_applicant({
    "person_age": 30,
    "person_income": 60000,
    "person_home_ownership": "RENT",
    "person_emp_length": 5,
    "loan_intent": "EDUCATION",
    "loan_amnt": 10000,
    "loan_percent_income": 0.17,
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 5,
})

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": clone(model),

    "Decision Tree": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(
            max_depth=6,
            min_samples_leaf=20,
            random_state=42
        )),
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )),
    ]),
}

for name, candidate in models.items():
    candidate.fit(X_fit, y_fit)
    probabilities = candidate.predict_proba(X_val)[:, 1]
    predictions = (probabilities >= 0.3).astype(int)

    print(
        f"{name}: "
        f"accuracy={accuracy_score(y_val, predictions):.3f}, "
        f"ROC-AUC={roc_auc_score(y_val, probabilities):.3f}, "
        f"precision={precision_score(y_val, predictions):.2f}, "
        f"recall={recall_score(y_val, predictions):.2f}"
    )

rf_probabilities = models["Random Forest"].predict_proba(X_val)[:, 1]

for threshold in [0.5, 0.4, 0.3, 0.2]:
    rf_predictions = (rf_probabilities >= threshold).astype(int)

    print(
        f"Threshold {threshold}: "
        f"accuracy={accuracy_score(y_val, rf_predictions):.3f}, "
        f"precision={precision_score(y_val, rf_predictions):.2f}, "
        f"recall={recall_score(y_val, rf_predictions):.2f}"
    )
final_model = clone(models["Random Forest"])
final_model.fit(X_train, y_train)

test_probabilities = final_model.predict_proba(X_test)[:, 1]
chosen_threshold = 0.3
test_predictions = (test_probabilities >= chosen_threshold).astype(int)

print(classification_report(
    y_test, test_predictions, target_names=["No default", "Default"]
))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))
print("ROC-AUC:", roc_auc_score(y_test, test_probabilities))

import joblib

artifact = {
    "model": final_model,
    "threshold": chosen_threshold,
    "features": features,
}

joblib.dump(artifact, "credit_default_random_forest.joblib")


Rows over $500,000: 53
(26064, 9) (6517, 9)
              precision    recall  f1-score   support

  No default       0.85      0.96      0.90      5095
     Default       0.75      0.39      0.51      1422

    accuracy                           0.84      6517
   macro avg       0.80      0.68      0.71      6517
weighted avg       0.83      0.84      0.82      6517

Confusion matrix:
[[4906  189]
 [ 867  555]]
ROC-AUC: 0.8049893100016701


c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\A

Threshold 0.5: precision=0.78, recall=0.39
Threshold 0.4: precision=0.66, recall=0.49
Threshold 0.3: precision=0.53, recall=0.58
Threshold 0.2: precision=0.41, recall=0.69
              precision    recall  f1-score   support

  No default       0.88      0.85      0.87      5095
     Default       0.53      0.60      0.56      1422

    accuracy                           0.80      6517
   macro avg       0.71      0.73      0.72      6517
weighted avg       0.81      0.80      0.80      6517

Confusion matrix:
[[4346  749]
 [ 569  853]]


c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\A

Logistic Regression: accuracy=0.795, ROC-AUC=0.797, precision=0.53, recall=0.58
Decision Tree: accuracy=0.868, ROC-AUC=0.836, precision=0.80, recall=0.52


c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\A

Random Forest: accuracy=0.866, ROC-AUC=0.867, precision=0.72, recall=0.63
Threshold 0.5: accuracy=0.879, precision=0.94, recall=0.48
Threshold 0.4: accuracy=0.876, precision=0.83, recall=0.54
Threshold 0.3: accuracy=0.866, precision=0.72, recall=0.63
Threshold 0.2: accuracy=0.814, precision=0.55, recall=0.73


c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Jaksh\A

              precision    recall  f1-score   support

  No default       0.90      0.94      0.92      5095
     Default       0.75      0.65      0.69      1422

    accuracy                           0.87      6517
   macro avg       0.83      0.79      0.81      6517
weighted avg       0.87      0.87      0.87      6517

Confusion matrix:
[[4783  312]
 [ 503  919]]
ROC-AUC: 0.8722715659846875
c:\Users\Jaksh\AI-ML-Projects\machine-learning\.venv\Scripts\python.exe
